# 00b - Crea e usa un tool

Nel notebook `00` abbiamo chiamato direttamente il modello. Prima di passare agli agenti con dati reali, facciamo un esempio completamente autonomo: un piccolo tool Python per stimare il costo di una spedizione.

Non usiamo dati del campus e non usiamo la cassetta degli attrezzi del notebook `01`. Qui il punto e capire il flusso: funzione Python, schema del tool, richiesta del modello, esecuzione del tool, risposta finale.


## Obiettivi

- definire un tool con `@tool`
- ispezionare nome, descrizione e argomenti del tool
- chiamare il tool direttamente
- dare il tool al modello con `bind_tools`
- eseguire manualmente una tool call e far produrre una risposta finale


In [1]:
import os
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_ollama import ChatOllama

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / ".env").exists() and (PROJECT_ROOT.parent / ".env").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

load_dotenv(PROJECT_ROOT / ".env", override=False)


def get_lab_model(provider: str = "ollama_cloud", model: str | None = None, temperature: float = 0):
    provider = provider.lower()
    use_cloud = provider in {"ollama_cloud", "cloud_ollama"}

    model = model or os.getenv("OLLAMA_CLOUD_MODEL" if use_cloud else "OLLAMA_MODEL")
    model = model or ("gemma3:latest" if use_cloud else "llama3.1:8b")

    kwargs = {
        "model": model,
        "temperature": temperature,
        "disable_streaming": "tool_calling",
    }

    base_url = os.getenv("OLLAMA_CLOUD_BASE_URL" if use_cloud else "OLLAMA_BASE_URL")
    if base_url:
        kwargs["base_url"] = base_url
    elif use_cloud:
        kwargs["base_url"] = "https://ollama.com"

    if use_cloud:
        api_key = os.getenv("OLLAMA_API_KEY")
        if not api_key:
            raise ValueError("OLLAMA_API_KEY e richiesta quando provider='ollama_cloud'. Inseriscila in .env.")
        headers = {"Authorization": f"Bearer {api_key}"}
        kwargs["client_kwargs"] = {"headers": headers}
        kwargs["async_client_kwargs"] = {"headers": headers}

    return ChatOllama(**kwargs)


print("OLLAMA_MODEL:", os.getenv("OLLAMA_MODEL", "(default llama3.1:8b)"))
print("OLLAMA_CLOUD_MODEL:", os.getenv("OLLAMA_CLOUD_MODEL", "(default gemma3:latest)"))
print("Chiave cloud presente:", "si" if os.getenv("OLLAMA_API_KEY") else "no")


OLLAMA_MODEL: (default llama3.1:8b)
OLLAMA_CLOUD_MODEL: gemma3:latest
Chiave cloud presente: si


## Scegli il backend

Usa `ollama` se vuoi inferenza locale. Usa `ollama_cloud` se vuoi usare la chiave cloud. Il resto del notebook non dipende da file dati o helper del progetto.


In [7]:
PROVIDER = "ollama"        # prova anche: "ollama"
MODEL_NAME = None

llm = get_lab_model(provider=PROVIDER, model=MODEL_NAME, temperature=0)


## Crea un tool minimo

Questo tool calcola un preventivo di spedizione. E volutamente piccolo, deterministico e completamente locale: non chiama API esterne e non legge file.


In [8]:
@tool
def shipping_quote(destination_country: str, weight_kg: float, express: bool = False) -> str:
    """Calcola un preventivo di spedizione in euro per un pacco internazionale."""
    country = destination_country.strip().lower()
    base_by_country = {
        "italia": 5.0,
        "francia": 9.0,
        "germania": 10.0,
        "spagna": 11.0,
        "stati uniti": 24.0,
    }
    base = base_by_country.get(country, 18.0)
    variable = max(weight_kg, 0.1) * 2.4
    express_fee = 8.0 if express else 0.0
    total = base + variable + express_fee
    speed = "express" if express else "standard"
    return f"Preventivo {speed} per {destination_country}: EUR {total:.2f}"


## Ispeziona lo schema

Il modello non vede il corpo della funzione. Vede soprattutto nome, descrizione e schema degli argomenti. Se questi sono ambigui, anche un buon modello puo scegliere male.


In [9]:
print("Nome:", shipping_quote.name)
print("Descrizione:", shipping_quote.description)
print("Argomenti:", shipping_quote.args)


Nome: shipping_quote
Descrizione: Calcola un preventivo di spedizione in euro per un pacco internazionale.
Argomenti: {'destination_country': {'title': 'Destination Country', 'type': 'string'}, 'weight_kg': {'title': 'Weight Kg', 'type': 'number'}, 'express': {'default': False, 'title': 'Express', 'type': 'boolean'}}


## Prova il tool direttamente

Prima regola pratica: testare il tool senza agente. Se il risultato non e utile qui, non diventera utile dentro un agente.

Chiamiamo la funzione con invoke passandogli un dizionario chiave valore di argomenti.


In [10]:
print(shipping_quote.invoke({
    "destination_country": "Germania",
    "weight_kg": 3.2,
    "express": True,
}))


Preventivo express per Germania: EUR 25.68


## Dai il tool al modello

`bind_tools` non esegue il tool. Aggiunge lo schema alla chiamata al modello. Il modello puo quindi decidere di rispondere direttamente oppure produrre una richiesta di tool call.


In [11]:
# Fa il binding dei tools. Stiamo dicendo all LLM: guarda ti do la disponibiltà di chiamare questi tool.
# Li guarda, legge descrizione argomenti etc e da ora in poi sapra che esistono.
llm_with_tool = llm.bind_tools([shipping_quote])

messages = [
    SystemMessage(content=(
        "Sei un assistente per spedizioni. Rispondi in italiano. "
        "Quando l'utente chiede un preventivo, usa il tool disponibile."
    )),
    HumanMessage(content="Quanto costa spedire in Germania un pacco da 3.2 kg con consegna express?"),
]

ai_msg = llm_with_tool.invoke(messages)

print("Contenuto:", ai_msg.content)
print("Tool calls:", ai_msg.tool_calls)

#Vediamo che l'output di questa cella non è tanto una risposta testuale
# ma la chiamata al tool con i parametri passati dalla query in Natural Language.


Contenuto: 
Tool calls: [{'name': 'shipping_quote', 'args': {'destination_country': 'Germania', 'express': True, 'weight_kg': 3.2}, 'id': '7a7a03c9-dcfc-4a00-b80f-bbac266aba8c', 'type': 'tool_call'}]


## Esegui la tool call

Questa e la parte che spesso resta nascosta nei framework agentici. Se il modello chiede un tool, il nostro codice deve eseguirlo, aggiungere un `ToolMessage` alla conversazione e richiamare il modello per la risposta finale.


In [12]:
tool_by_name = {shipping_quote.name: shipping_quote}
conversation = messages + [ai_msg]

if not ai_msg.tool_calls:
    print("Il modello non ha chiesto tool. Prova a rendere la domanda piu concreta o il system prompt piu esplicito.")
else:
    for call in ai_msg.tool_calls:
        selected_tool = tool_by_name[call["name"]]
        tool_result = selected_tool.invoke(call["args"])
        #Passiamo alla conversazione il risultato del tool.
        conversation.append(ToolMessage(content=tool_result, tool_call_id=call["id"]))
        print(f"Risultato di {call['name']}:")
        print(tool_result)

    # Chiamo la LLM arricchita.
    final_msg = llm_with_tool.invoke(conversation)
    print("\nRisposta finale:")
    print(final_msg.content)


Risultato di shipping_quote:
Preventivo express per Germania: EUR 25.68

Risposta finale:
Il preventivo per spedire un pacco da 3,2 kg in Germania con consegna espressa è di circa 25,68 euro.


## Mini-sfida

Cambia una sola cosa e riesegui le celle:

- il nome del tool
- la descrizione del tool
- il nome di un argomento, per esempio `express`
- la domanda dell'utente

Poi osserva se il modello continua a scegliere il tool giusto. La descrizione del tool e parte del prompt, anche se vive nel codice.


In [13]:
# Spazio di lavoro: prova una variante del tool o una domanda diversa.
# Esempio: cambia paese, peso o consegna express/standard.

nuova_domanda = "Mi fai un preventivo standard per spedire 1.5 kg in Francia?"

variant_msg = llm_with_tool.invoke([
    SystemMessage(content="Usa il tool quando la domanda richiede un preventivo di spedizione."),
    HumanMessage(content=nuova_domanda),
])

print("Tool calls:", variant_msg.tool_calls)
print("Contenuto:", variant_msg.content)


Tool calls: [{'name': 'shipping_quote', 'args': {'destination_country': 'Francia', 'express': False, 'weight_kg': 1.5}, 'id': 'e4f6353e-c8f6-47e0-91b9-2bef57262eed', 'type': 'tool_call'}]
Contenuto: 


## Cosa portare nel notebook 01

Ora il flusso e completo ma ancora manuale:

1. definisci un tool Python
2. dai lo schema al modello
3. leggi la tool call
4. esegui il tool
5. rimandi il risultato al modello

Nel notebook `01` useremo lo stesso meccanismo tecnico, ma con tool legati ai dati del lab e con un loop LangGraph che automatizza questi passaggi.
